In [1]:
!pip install wrds pandas matplotlib streamlit ipykernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 1.9 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 1.9 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [pandas]
      Successfully uninstalled pandas-2.3.3━━━━━━━━━━━━━━━━━━━ 1/3 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [wrds]1/3 [pandas]


In [3]:
import wrds
import pandas as pd
db = wrds.Connection(
    wrds_username="你的WRDS用户名",
    wrds_password="你的WRDS密码"
)

Enter your WRDS username [你的WRDS用户名]: haoxiadong
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  tech_stocks
Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
def get_stock_finance(ticker, start=2020, end=2024):
    query = f"""
    SELECT datadate, fyear, conm, tic,
           at, lt, sale, ni, che, rect
    FROM comp.funda
    WHERE tic = '{ticker}'
    AND fyear BETWEEN {start} AND {end}
    AND indfmt='INDL' AND datafmt='STD'
    ORDER BY fyear
    """
    df = db.raw_sql(query)

    # 计算财务比率
    df['ROE'] = df['ni'] / (df['at'] - df['lt'])
    df['资产负债率'] = df['lt'] / df['at']
    df['销售净利率'] = df['ni'] / df['sale']
    df['营收增长率'] = df['sale'].pct_change()
    
    return df

# 测试：获取苹果公司数据
df = get_stock_finance("AAPL")
print(df)

     datadate  fyear       conm   tic        at        lt      sale       ni  \
0  2020-09-30   2020  APPLE INC  AAPL  323888.0  258549.0  274515.0  57411.0   
1  2021-09-30   2021  APPLE INC  AAPL  351002.0  287912.0  365817.0  94680.0   
2  2022-09-30   2022  APPLE INC  AAPL  352755.0  302083.0  394328.0  99803.0   
3  2023-09-30   2023  APPLE INC  AAPL  352583.0  290437.0  383285.0  96995.0   
4  2024-09-30   2024  APPLE INC  AAPL  364980.0  308030.0  391035.0  93736.0   

       che     rect       ROE     资产负债率     销售净利率     营收增长率  
0  90979.0  37445.0  0.878664  0.798267  0.209136      <NA>  
1  62639.0  51506.0  1.500713  0.820257  0.258818  0.332594  
2  48304.0  60932.0  1.969589  0.856354  0.253096  0.077938  
3  61555.0  60985.0   1.56076  0.823741  0.253062 -0.028005  
4  65171.0  66243.0  1.645935  0.843964  0.239713   0.02022  
